# 《蔬菜类商品自动定价与补货决策》授课导读手册

> **配套文件**：`蔬菜类商品自动定价与补货决策_2h授课课件.ipynb`（89 单元 = 33 讲解 + 56 代码，含 38 图 / 84 表）
>
> **本手册用途**：讲师备课与学生课后复盘。逐一说明主课件里**每个代码块在干什么**、**每张图怎么读**、**每张表说明什么结论**。
>
> 阅读方式：左边开主课件，右边开本手册，按编号对照。代码块编号 `[00]~[55]` 与主课件中代码单元的先后顺序一致。

# 第一章　题目到底在问什么

## 1.1 赛题来源与背景

2023 年全国大学生数学建模竞赛（CUMCM）**C 题：蔬菜类商品的自动定价与补货决策**。

商超经营生鲜蔬菜有三个绕不开的现实约束：

1. **保鲜期极短**。蔬菜品相随时间快速下降，大部分品类**当日未售出、隔日就无法再以正常品质售出**。这一句话是全课最重要的建模前提——它把问题锁定为**单周期（single-period）**决策，而不是可以跨期结转库存的多周期问题。
2. **补货在"知道需求之前"发生**。商超通常凌晨 3:00–4:00 就要向批发市场下单，而当天的真实客流与销量要到晚上才知道。所以**订货量必须在需求实现之前确定**，这是典型的"先决策、后观测"结构。
3. **品类多、单品杂、进货价天天变**。可售单品常有 100 种以上，不同单品的产地、运输、时令都不同，批发价逐日波动。

## 1.2 四个问题的递进关系

| 问题 | 决策对象 | 时间范围 | 本质 |
|---|---|---|---|
| **问题一** | 无决策，纯分析 | 2020-07 ~ 2023-06 | **描述**：品类与单品的销量分布规律及相互关系 |
| **问题二** | 6 个**品类**的补货量与定价 | 2023-07-01 ~ 07-07（7 天） | **优化**：品类层面的收益最大化 |
| **问题三** | 单品的**选品 + 补货量 + 定价** | 2023-07-01（1 天） | **带约束优化**：27~33 个 SKU、最小陈列量 2.5 kg |
| **问题四** | 无决策，提建议 | —— | **反思**：还需要采集什么数据 |

四问是一条完整的建模链：**问题一摸清规律 → 问题二在品类层面建立"需求—定价—利润"闭环 → 问题三把粒度下沉到单品并加入离散选品约束 → 问题四回头审视模型的信息缺口**。

## 1.3 题目原文中最容易被忽略的三句话

授课时务必强调，这三句话直接决定模型形式：

1. **"商超通常以品类为单位做补货计划"** → 问题二的决策变量是品类级的，不能直接堆单品。
2. **"当日未售出的商品隔日就无法再售"** → 单周期、**残值为零**。这一点后面会用数据反复验证（代码块 `[06]` 与 `[36]`），也是本课最大的一个"坑"。
3. **"可售单品总数控制在 27~33 个，且各单品订购量不少于 2.5 千克"** → 问题三必须是**整数规划**，不能只用连续优化。

# 第二章　四份附件：逐字段说明

主课件代码块 `[02]` 负责把这四张表读进来。这里先讲清每一份数据是什么。

## 2.1 附件1：商品信息表（`附件1.xlsx`）

**规模**：251 行 × 4 列。这是一张**维度表（dimension table）**，起"字典"作用。

| 字段 | 类型 | 含义 | 建模用途 |
|---|---|---|---|
| `单品编码` | int64 | 单品唯一 ID，如 `102900005115168` | **主键**，与附件2、附件3关联 |
| `单品名称` | str | 如"云南生菜(份)"、"净藕(1)" | 结果表可读性；名称里的"(份)""(1)"暗示包装规格差异 |
| `分类编码` | int64 | 品类 ID | 与 `分类名称` 一对一 |
| `分类名称` | str | **6 个品类**：花叶类、花菜类、水生根茎类、茄类、辣椒类、食用菌 | 问题二的决策单位；问题三的分组约束 |

**教学要点**：251 个单品 → 6 个品类，是**多对一**关系。品类是"花叶类"这种大类，不是"生菜"这种细类。六个品类的单品数极不均衡（花叶类最多），这个不均衡会直接影响问题三的选品结构。

## 2.2 附件2：销售流水明细（`附件2.xlsx`）

**规模**：约 **87.8 万行** × 7 列，文件 39 MB，是四份数据里唯一的**事实表（fact table）**，也是全课的核心数据源。时间跨度 **2020-07-01 ~ 2023-06-30**，整三年。

| 字段 | 类型 | 含义 | 关键陷阱 |
|---|---|---|---|
| `销售日期` | datetime | 交易日期 | 聚合到"日"是本题的天然粒度 |
| `扫码销售时间` | time | 精确到秒的时点 | **购物篮分析的依据**：同一顾客的一次结账，多个商品的扫码时间相差仅几秒 |
| `单品编码` | int64 | 外键 → 附件1 | 需校验完整性 |
| `销量(千克)` | float | 本笔交易的重量 | 注意是**重量**不是件数，蔬菜按 kg 计价 |
| `销售单价(元/千克)` | float | 本笔**实际成交**单价 | 已含折扣，不是标价 |
| `销售类型` | str | `销售` / `退货` | **退货必须处理**，否则需求被高估 |
| `是否打折销售` | str | `是` / `否` | 打折是"品相下降后降价促销"的信号 |

**三个必须在课上讲清的处理决策**：

1. **退货怎么办**：退货行的销量为正但业务含义是负。本课采用**净销量**口径（销售 − 退货），因为我们要估计的是"真实带走的量"。
2. **打折行怎么办**：不能删。打折成交也是真实销量，删掉会低估需求。但它的**价格**不能直接用来估需求曲线，否则会把"因为快坏了所以降价"误读成"降价导致销量上升"。
3. **单价为什么每笔都不同**：同一单品同一天可能有多个价格（正价 + 若干折扣价）。因此计算"当日代表价格"必须用**销量加权平均价**，而不是简单平均：

$$\bar p_{it}=\frac{\sum_k p_{itk}\,q_{itk}}{\sum_k q_{itk}}$$

其中 $k$ 遍历该单品该日的所有交易笔次。简单平均会让一笔 0.1 kg 的折扣交易和一笔 50 kg 的正价交易权重相同，严重失真。

## 2.3 附件3：批发价明细（`附件3.xlsx`）

**规模**：约 5.5 万行 × 3 列。时间跨度与附件2 基本一致。

| 字段 | 类型 | 含义 | 建模用途 |
|---|---|---|---|
| `日期` | datetime | 日期 | 与附件2 按 (日期, 单品) 关联 |
| `单品编码` | int64 | 外键 | —— |
| `批发价格(元/千克)` | float | 当日进货成本 $w_{it}$ | **成本基准**；也是需求函数的**工具变量** |

**教学要点**：批发价是本题唯一的**外生成本变量**。它有两个身份：

- **身份一（成本）**：算利润必须用它。
- **身份二（工具变量）**：估计需求弹性时，零售价 $p$ 与需求 $Q$ 存在**联立内生性**（商超看到需求旺就抬价，价格与误差项相关，OLS 有偏）。批发价 $w$ 影响 $p$ 但不直接影响消费者需求，是天然的工具变量。这是问题二能做得比别人深的关键。

## 2.4 附件4：损耗率（`附件4.xlsx`）

**规模**：很小的一张表，但**注意它有两个 sheet**，且损耗率是**品类级**（不是单品级）的。

| 字段 | 含义 |
|---|---|
| `小分类名称` | 品类名 |
| `损耗率(%)` | 该品类的平均损耗率 $\theta_c$，约 8%~15% |

**损耗率怎么进模型 —— 这是最容易算错的地方**。损耗指的是"进货 100 kg，因挑拣、磕碰、失水，只有 $(1-\theta)\times 100$ kg 能上架卖"。所以：

- 若订货 $x$ kg，**可售量**为 $x(1-\theta)$；
- 若想卖出 $S$ kg，**必须订** $x = S/(1-\theta)$ kg；
- 因此单位可售商品的**有效成本**是

$$c=\frac{w}{1-\theta}$$

**常见错误**：把损耗当成"卖不掉的部分"，用 $c = w(1+\theta)$ 或直接从利润里减掉 $\theta$。前者数值上接近但概念错误（$\frac{1}{1-\theta}\neq 1+\theta$），后者会把损耗和滞销混为一谈。主课件代码块 `[08]` 专门用一张表对比三种口径的差别，务必在课上讲。

# 第三章　方法总览：每道题用了什么，为什么用

本课的设计原则是**每道题都给多种方法并横向比较**，让学生看到"方法选择本身也要论证"。

## 3.1 数据工程阶段（第一部分）

| 方法 | 解决什么 |
|---|---|
| 主键唯一性 / 外键完整性校验 | 建模前的"体检"，防止后面出现莫名的 NaN |
| 净销量口径（销售 − 退货） | 得到真实需求 |
| 销量加权平均价 | 得到无偏的"当日代表价格" |
| 打折力度 $\delta$ 的经验估计 | 为报童模型的残值提供数据依据 |
| 打折盈亏检验（$\rho$ vs $1/(1-\theta)$） | **推翻"打折=亏本清仓"的想当然假设** |
| 有效成本 $c=w/(1-\theta)$ | 正确处理损耗 |

## 3.2 问题一：七种分析方法

| # | 方法 | 回答什么问题 | 代码块 |
|---|---|---|---|
| 1 | 描述性统计 + 帕累托（80/20）分析 | 销量集中在哪些单品？ | `[10] [11]` |
| 2 | 六种概率分布拟合（KS + AIC/BIC） | 日销量服从什么分布？ → 为报童模型选分布 | `[12] [13]` |
| 3 | STL 时间序列分解 | 趋势、季节、残差各占多少方差？ | `[14]` |
| 4 | ANOVA + Kruskal-Wallis 双检验 | 周内效应显著吗？（参数 + 非参数互相印证） | `[15] [16]` |
| 5 | Pearson / Spearman / Kendall + 偏相关 | 品类间是真关联还是被共同趋势带的伪相关？ | `[17]` |
| 6 | 层次聚类 / K-means / DBSCAN + PCA | 单品能分成几种"性格"？ | `[18] [19] [20]` |
| 7 | Apriori 关联规则 + ADF + Granger 因果 | 哪些品类会被一起买？谁领先谁？ | `[21] [22] [23] [24]` |

## 3.3 问题二：品类级补货与定价

**核心是三步：估需求 → 预测成本 → 优化利润。**

**第一步，需求函数（七种规格横向比较）**：

| 规格 | 形式 | 特点 |
|---|---|---|
| M1 线性 | $Q=a+bp$ | 最简单，但最优价常常荒谬 |
| M2 双对数 | $\ln Q=a+\varepsilon\ln p$ | $\varepsilon$ 直接就是弹性，最常用 |
| M3 半对数 | $\ln Q=a+bp$ | —— |
| M4 二次型 | $Q=a+bp+cp^2$ | 允许非单调 |
| M5 加控制变量 | $\ln Q=a+\varepsilon\ln p+\gamma\ln w+\text{星期}+\text{月份}+t$ | 控制混淆因素 |
| M6 IV / 2SLS | 用 $w$ 做工具变量 | 处理价格内生性 |
| **M7 一阶差分** | $\Delta\ln Q=\varepsilon\,\Delta\ln p+\cdots$ | **本课采用**：差分掉所有慢变因素，6/6 品类弹性为负且显著 |

**为什么最后选 M7**：代码块 `[27]` 的结果表显示，只有一阶差分做到了**六个品类的弹性全部为负且显著**。其余规格常出现正弹性（违背经济学常识），根源是季节、趋势等慢变混淆因素没被清除。而且模型选择**不能只看 $R^2$** —— 代码块 `[28]` 用滚动时序交叉验证（rolling-origin CV）比较真实预测误差，这才是有说服力的依据。

**第二步，批发价预测（四种方法回测）**：移动平均、Holt-Winters、SARIMA、随机森林，用滚动回测比 MAE/RMSE/MAPE 后择优（代码块 `[31] [32]`）。

**第三步，报童模型 + 定价联合优化**。设定价 $p$、有效成本 $c=w/(1-\theta)$、残值 $s=0$，则最优服务水平（临界分位数）为

$$q^\*=\frac{C_u}{C_u+C_o}=\frac{p-c}{(p-c)+(c-s)}=\frac{p-c}{p-s}\;\xrightarrow{\;s=0\;}\;1-\frac{c}{p}$$

订货量取需求分布的该分位数：$x^\*=F^{-1}(q^\*)/(1-\theta)$。定价 $p$ 则在**历史加成率分位数构成的可行域（trust region）**内做网格搜索 + 数值优化，避免外推到数据从未覆盖的价格区间。

## 3.4 问题三：单品选品 + 补货 + 定价

| 方法 | 作用 | 代码块 |
|---|---|---|
| 经验贝叶斯收缩 | 单品数据少、弹性估计噪声大 → 向品类均值收缩 | `[42]` |
| (单品 × 价格档) 利润矩阵 | 把连续定价离散化，为整数规划做准备 | `[44]` |
| 贪心 / 排序法 | 只有基数约束时是全局最优（**均匀拟阵**性质） | `[45]` |
| **混合整数线性规划（MILP）** | 同时处理 SKU 数、最小陈列量、品类覆盖 | `[46]` |
| 遗传算法 | 独立验证 MILP 的解，交叉印证 | `[47]` |
| $\alpha$ 覆盖强度扫描 | 把"尽量满足市场需求"翻译成可计算约束 | `[48]` |
| 约束推紧对比实验 | 证明贪心在非拟阵结构下失效 | `[49]` |

**经验贝叶斯收缩公式**（问题三的技术亮点）：单品 $i$ 的弹性取

$$\hat\varepsilon_i=\lambda_i\hat\varepsilon_i^{\text{OLS}}+(1-\lambda_i)\hat\varepsilon_{c(i)},\qquad
\lambda_i=\frac{\tau^2}{\tau^2+\sigma_i^2}$$

$\sigma_i^2$ 是单品自身估计的方差（数据少 → 大），$\tau^2$ 是品类内弹性的真实离散度。**数据越少、估计越不稳，就越向品类均值靠**。本课额外设 $\lambda\le 0.8$，保证品类信息至少占 20% 权重。

**MILP 形式化**：令 $y_{ij}\in\{0,1\}$ 表示"单品 $i$ 采用价格档 $j$"，

$$\max\sum_{i,j}\pi_{ij}y_{ij}$$

约束：(C1) $27\le\sum_{i,j}y_{ij}\le33$；(C2) 每个单品至多一个价格档 $\sum_j y_{ij}\le1$；(C3) 最小陈列量 $x_i\ge2.5$ kg；(C5) 每品类至少上架若干单品；(C6) 每品类可售量 $\ge\alpha\bar D_c$。

## 3.5 问题四：数据采集建议

不空谈，而是**用模型残差说话**：对需求模型做逐步方差分解，量化"价格解释了多少、季节解释了多少、还有多少无法解释"，再据此按"能填哪个坑"组织数据需求清单，并画价值—难度四象限图（代码块 `[52] [53] [54]`）。

# 第四章　逐代码块导读（上）：环境、数据工程、问题一

> 每条包含四栏：**干什么** / **怎么读输出** / **图表含义** / **讲课提示**。

## 第 0 部分　课程框架与环境

### `[00]` 环境配置
- **干什么**：导入库；按可用性回退选择中文字体；设置绘图风格与色盲安全调色板（Okabe-Ito 6 色）；自动定位附件目录；定义全课通用工具函数 `show()`（显示表）、`dump()`（导出 CSV）、`savefig()`（存图）。
- **怎么读输出**：打印 `中文字体 -> Microsoft YaHei` 表示中文不会显示成方框；`附件目录 -> ...` 确认数据找到了。
- **讲课提示**：`axes.unicode_minus=False` 这一行是修复负号显示成方框的关键，很多同学的图上"-0.5"变成"口0.5"就是漏了它。

### `[01]` 建模总蓝图
- **干什么**：现场用 matplotlib 画出四问的逻辑关系与数据流向。
- **图 `00_建模总蓝图`**：左侧四份附件 → 中间数据工程层 → 右侧四个问题模块，箭头表示依赖。
- **讲课提示**：这张图可直接放进论文当"模型总体框架图"。建议开场就展示，让学生先有全局观。

### `[02]` 数据加载
- **干什么**：读四份附件，附件2 因为 39 MB 做本地 pickle 缓存（首次 1~2 分钟，之后秒开）。
- **4 张表**：四份附件各自的前几行预览。
- **讲课提示**：缓存这个工程习惯值得强调 —— 比赛三天，反复重读大 Excel 是纯浪费时间。

## 第 1 部分　数据工程

### `[03]` 主键唯一性与外键完整性校验
- **表 1-1　数据体检报告**：逐项检查行数、缺失、重复、日期范围。
- **表 1-2　从未销售的单品**：这些单品在附件1 里有、但流水里从没出现 → **问题三的候选集必须剔除**。
- **表 1-3　各品类单品数**：暴露六品类严重不均衡。
- **讲课提示**：这是"建模前的体检"。不做体检就建模，后面出现莫名 NaN 会浪费大量时间却找不到原因。

### `[04]` 执行清洗
- **表 1-4　数据清洗日志**：每一步"删了多少行、为什么删、剩多少"。
- **讲课提示**：**这张表要原样搬进论文**。清洗过程可复现是评审很看重的一点，只写"我们清洗了数据"是拿不到分的。

### `[05]` 估计打折力度 $\delta$
- **干什么**：用打折成交价 / 同期正价，估计折扣比例 $\delta\approx0.6$。
- **表 1-5 + 图 `01_打折力度`**：各品类 $\delta$ 的分布，打印"打折成交价约为正价的 60%"。
- **讲课提示**：这个 $\delta$ 是从数据里**估**出来的，不是拍脑袋假设的。但下一个代码块会说明：$\delta$ **不能**直接当报童模型的残值率。

### `[06]` ★ 打折销售究竟是"亏本清仓"还是"正常销售"
- **干什么**：本课**最重要的判别实验**。计算 $\rho=$ 打折成交价 / 当日批发价，与有效成本系数 $1/(1-\theta)$ 比较。
- **表 1-6 + 图 `01b_打折盈亏检验`**：结果 $\rho\approx1.26$，而 $1/(1-\theta)\in[1.07,1.18]$ → **打折价仍高于成本**。
- **结论含义**：数据里的"打折"不是亏本甩卖，而是品相下降后的降价促销，**它卖出的量本身已计入当日需求**。真正的损失是完全没卖掉、隔日报废的部分。因此报童模型的残值应取 $s=0$（正合题面"隔日就无法再售"）。
- **讲课提示**：这是全课的**方法论高潮之一** —— 用数据推翻一个想当然的假设。如果误取 $s=\delta p$，会引发代码块 `[36]` 演示的"永动机"灾难。

### `[07]` 构建日面板
- **干什么**：把 87.8 万行流水聚合成【单品日面板】与【品类日面板】，这是后续所有建模的基础数据结构。
- **表 1-6/1-7**：两个面板的样例。
- **讲课提示**：强调价格用**销量加权平均**，不是简单平均。

### `[08]` 三种损耗口径对比
- **表 1-8 + 图 `02_损耗口径`**：对比 $c=w/(1-\theta)$（正确）、$c=w(1+\theta)$（近似但概念错）、直接扣减（错）。
- **讲课提示**：$\frac{1}{1-\theta}$ 与 $1+\theta$ 在 $\theta$ 小时数值接近，所以错了也不容易被发现，但概念必须讲对。

### `[09]` 数据质量总览
- **表 1-9 + 图 `03_面板总览`**：各品类的数据覆盖天数、缺失率、销量与价格区间。
- **讲课提示**：确认六品类都有足够长的历史，可以放心建模。

## 第 2 部分　问题一：七种分析方法

### `[10] [11]` 方法1：描述统计 + 帕累托分析
- **表 2-1**：六品类日销量的均值/中位数/标准差/偏度/峰度。**偏度全为正 → 右偏 → 不能用正态分布**，这是后面选对数正态/伽马的依据。
- **表 2-2 + 图 `04_帕累托`**：销量 Top15 单品；累积曲线显示少数单品贡献大部分销量。
- **讲课提示**：右偏这个观察要和 `[12]` 的分布拟合、`[35]` 的报童分位数串起来讲，形成逻辑链。

### `[12] [13]` 方法2：六种概率分布拟合
- **表 2-3/2-4**：六分布（正态、对数正态、伽马、威布尔、指数、泊松）× 六品类的 KS 统计量与 AIC/BIC，并给出每品类的最优分布。
- **图 `05_分布拟合`**：直方图 + 前三名密度曲线叠加。**图 `06_QQ图`**：对数正态的 Q-Q 图，点贴近 45° 线说明拟合好。
- **讲课提示**：这不是"炫技"，报童模型需要 $F^{-1}(q)$，**必须先知道 $F$ 是什么**。方法与目的的衔接要讲明。

### `[14]` 方法3：STL 时间序列分解
- **表 2-5 + 图 `07_STL`**：把 $\ln Q$ 拆成趋势 + 季节 + 残差，并给出各成分的方差占比。
- **讲课提示**：取对数再分解，是因为销量右偏且季节效应是**乘性**的（旺季按比例放大，不是加固定值）。

### `[15] [16]` 方法4：周内效应与年度季节性
- **表 2-6/2-7/2-8 + 图 `08_周内效应`**：各品类分星期的平均销量、周内需求指数，以及 **ANOVA（参数）与 Kruskal-Wallis（非参数）双检验**。
- **表 2-9/2-10 + 图 `09_月度季节`**：月度需求指数热力图，并验证"4–10 月批发价更低（供应丰富）"。
- **讲课提示**：为什么做两种检验？ANOVA 要求正态与方差齐性，而销量右偏，所以补一个不依赖分布的 Kruskal-Wallis。**两者结论一致，才敢下断言**。这个"稳健性检验"思路评审很喜欢。

### `[17]` 方法5：三种相关系数 + 偏相关
- **表 2-11/2-12 + 图 `10_相关矩阵` `11_偏相关`**：Pearson / Spearman / Kendall 三种相关矩阵，以及控制共同趋势后的**偏相关**。
- **关键结论**：原始 Pearson 相关全为正（0.11~0.69），看起来"品类间普遍互补"；但**偏相关后大量相关性显著减弱甚至反号** —— 说明原来的正相关主要是被**共同的季节与趋势**带出来的**伪相关**。
- **讲课提示**：这是问题一最有含金量的一段。"相关不等于关联"，会做偏相关就能和大多数队伍区分开。

### `[18] [19] [20]` 方法6：聚类 + PCA
- **表 2-13/2-14**：单品的聚类特征（销量规模、价格水平、波动性、季节强度等）及其描述统计。
- **表 2-15 + 图 `12_聚类`**：K-means 的簇数选择（轮廓系数 + 肘部法）与 PCA 二维可视化。
- **表 2-16/2-17**：**簇画像**（每簇的特征均值，用于解释每类单品的"性格"）与三种聚类方法（层次/K-means/DBSCAN）的一致性检验。
- **讲课提示**：聚类的价值不在于分出几类，而在于**簇画像能否讲出业务故事**（如"高销量低毛利的引流款"vs"低销量高毛利的特色款"）。这直接给问题三的选品提供结构化依据。

### `[21] [22] [23]` 方法7a：购物篮关联规则
- **表 2-18**：购物篮时间窗敏感性（为什么选 60 秒）。
- **表 2-19/2-20 + 图 `13_关联规则`**：品类级 Apriori 完整规则表（支持度/置信度/提升度），以及提升度最高的 15 条。
- **表 2-21/2-22**：单品级 Top25 规则；**同品类 vs 跨品类关联强度对比**。
- **讲课提示**：时间窗敏感性分析是加分项 —— 说明 60 秒不是随便选的。提升度 lift > 1 才叫真关联，只看支持度会被高频商品误导。

### `[24]` 方法7b：ADF + Granger 因果
- **表 2-23**：ADF 单位根检验 —— **这是 Granger 检验的前置条件**，非平稳序列做 Granger 会得到伪因果。
- **表 2-24 + 图 `14_互相关` `15_Granger`**：Granger 因果的最小 p 值矩阵（行=原因，列=结果）。
- **讲课提示**：务必强调 Granger 因果**只是"预测有用"，不是真正的因果**。先做 ADF 再做 Granger 这个顺序，是方法严谨性的体现。

### `[25]` 问题一结论汇总
- **干什么**：把七种方法的结论收敛成一张表。
- **讲课提示**：告诉学生问题一的答案不是"我画了很多图"，而是**"规律 + 相互关系"这两个词要各自有明确回答**。

# 第五章　逐代码块导读（下）：问题二、三、四

## 第 3 部分　问题二：品类级补货与定价

### `[26]` 销量 vs 加成率的原始关系
- **表 3-1 + 图 `16_加成率vs销量`**：散点 + 分位分组。
- **讲课提示**：先看原始关系再上模型。图上看不出干净的负斜率，正好引出"为什么需要控制变量和差分"。

### `[27]` ★ 七种需求函数规格横向比较
- **表 3-2**：七规格 × 六品类 = 42 行完整拟合结果（弹性、p 值、$R^2$）。**表 3-3**：哪种规格最优。
- **图 `17_规格对比`**：各规格下六品类弹性的取值分布。
- **关键结论**：**只有 M7（一阶差分）做到 6/6 品类弹性为负且显著**。差分把季节、趋势等慢变混淆因素全部消掉了。
- **讲课提示**：其余规格出现**正弹性**（涨价反而卖更多）不是"数据有问题"，而是**模型设定有问题** —— 遗漏变量偏误。这个诊断能力比会跑回归重要得多。

### `[28]` 滚动时序交叉验证
- **表 3-4 + 图 `18_CV对比`**：7 折 × 14 天预测窗的滚动 RMSE。
- **讲课提示**：**模型选择不能只看 $R^2$**。$R^2$ 是样本内拟合，加变量必然上升；真实预测能力必须用样本外验证。而且时序数据**不能用普通 K-fold**（会用未来预测过去，即数据泄漏），必须用 rolling-origin。

### `[29]` 加成率弹性的稳健估计
- **表 3-5 + 图 `19_弹性对比`**：M7 估计 + 经验贝叶斯收缩后的最终弹性，并给出收缩权重 $\hat\lambda$。
- **讲课提示**：收缩是"让噪声大的估计向稳定的均值靠"，是本课贯穿问题二、三的统一思想。

### `[30]` 需求基线（锚点）模型
- **表 3-6 + 图 `20_基线拟合`**：$\ln Q=\alpha+\gamma\ln w+\text{星期}+\text{月份}+\text{趋势}$ 的拟合与诊断。
- **讲课提示**：弹性决定"价格变动的影响"，锚点决定"基准水平在哪"。两者分开估计，是本课优化能跑得快的关键（详见 `[35]`）。

### `[31] [32]` 批发价预测：四方法回测 + 出预测
- **表 3-7**：四种方法（移动平均、Holt-Winters、SARIMA、随机森林）的总体回测表现。**表 3-8/3-9**：各品类 RMSE 与 MAPE 明细。
- **图 `21_批发价回测`**：滚动回测误差对比。
- **表 3-10 + 图 `22_批发价预测`**：**7 月 1–7 日各品类批发价预测（四方法并列）**，并标注采用哪一种。
- **讲课提示**：为什么简单方法常常赢？因为批发价接近随机游走，复杂模型容易过拟合。**"简单方法胜出"本身就是一个值得写进论文的发现**，不要因为它不酷就藏起来。

### `[33]` 解析最优解诊断
- **表 3-11**：三种需求形式下的解析最优解。
- **关键结论**：恒弹性模型在 $-1<\varepsilon<0$（**需求缺乏弹性**）时，利润对价格**单调递增 → 无内点解**，数学上"价格越高越赚"。
- **讲课提示**：这解释了为什么必须引入**可行域约束**。不能因为公式说"涨价就赚"就真去涨到天上 —— 那是模型在数据覆盖范围外的荒谬外推。

### `[34]` 定价可行域（trust region）
- **表 3-12 + 图 `23_可行域`**：各品类近一年加成率的 10%/90% 分位数构成的搜索区间。
- **讲课提示**："信任区域"的思想：**只在数据说过话的范围里做决策**。这是应对上一块无内点解问题的正规做法。

### `[35]` 报童 + 定价联合优化引擎
- **干什么**：全课的**计算核心**。给定 $(p,w,\theta,$ 需求分布$)$，先算临界分位数得订货量，再在可行域内搜索最优 $p$。
- **表 3-12a**：加成率与最优服务水平的关系（以 $\theta=12.83\%$ 的花叶类为例）。
- **核心公式**：$q^\*=1-c/p$，其中 $c=w/(1-\theta)$。
- **讲课提示**：这里有个漂亮的性质 —— **最优服务水平只取决于毛利率**，与需求分布的形状无关（分布只影响订货量的具体数值）。毛利越高 → 越应该多备货（缺货损失大于积压损失）。

### `[36]` ★ 演示"永动机"陷阱
- **干什么**：把残值误设为 $s=\delta p$，展示会发生什么。
- **表 3-12b + 图 `23b_永动机陷阱`**：对比 $s=\delta p$ 与 $s=0$。前者出现 $C_o=c-\delta p\le0$ —— **负的积压成本**。
- **为什么是灾难**：$p$ 是**决策变量**，优化器一抬价，残值跟着涨，模型于是认为"卖不掉打折还能赚"，最优订货量**发散**。症状是服务水平顶到 99.9%、补货量达到需求的 3~5 倍。
- **讲课提示**：这是本课**最值得讲的一个坑**。教学价值不在于给出正确答案，而在于展示**如何识别退化解**：看到服务水平顶在边界、参数扫描曲线全平，就要怀疑模型而不是接受结果。

### `[37]` 利润曲线可视化
- **图 `24_利润曲线`**：一图看懂"定价 → 需求 → 利润"的完整机制。

### `[38] [39]` ★ 问题二最终方案
- **表 3-13**：**问题二的答案** —— 6 品类 × 7 天的日补货总量与定价策略。
- **表 3-14**：三方案对比（A 恒弹性+报童 / B 二次型+报童 / C 恒弹性+确定型）。
- **表 3-15/3-16**：按日、按品类汇总。**图 `25_问题二方案`**。
- **怎么读**：方案 C（确定型，忽略不确定性）利润 8803 元 > 方案 A（报童）6102 元。**这不是 C 更好**，而是 C **系统性高估** —— 它假装需求已知，忽略了缺货与积压的真实成本。
- **讲课提示**：这个对比要讲透。"考虑不确定性会让账面利润变低，但那才是能实现的利润。"

### `[40]` 敏感性分析
- **表 3-17 + 图 `26_敏感性`**：三因素扰动（弹性、批发价、残值假设）的龙卷风图。
- **讲课提示**：龙卷风图按影响幅度排序，一眼看出哪个参数最该重视。论文里有敏感性分析是模型可靠性的必要证据。

## 第 4 部分　问题三：单品选品 + 补货 + 定价

### `[41]` 候选单品集合
- **表 4-1/4-2 + 图 `27_候选单品`**：用 6/24–6/30 的在售品种确定 **49 个候选单品**，要从中选 27~33 个。
- **讲课提示**：候选集怎么定要有依据。用"最近一周实际在售"比用"历史上卖过"合理得多 —— 三年前卖过的品种现在可能根本进不到货。

### `[42]` 单品弹性：OLS → 收缩 → 经济学裁剪
- **表 4-3**：单品弹性直接估计的**问题诊断**（符号错误、标准误巨大）。**表 4-4**：49 个单品的弹性估计全过程（OLS 值、方差、收缩权重、最终值）。
- **图 `28_单品弹性收缩`**：收缩前后对比，能看到离群估计被拉回。
- **讲课提示**：单品层面数据少，直接估计噪声极大，**这正是经验贝叶斯的用武之地**。裁剪到 $[-3.0,-0.15]$ 是加一层经济学常识约束。

### `[43]` 单品建模参数汇总
- **表 4-5**：49 单品的需求锚点、批发价预测、不确定性、可行域。**表 4-6**：品类需求基准 $\bar D_c$（约束 C6 用）。

### `[44]` (单品 × 价格档) 利润矩阵
- **表 4-7 + 图 `29_单品盈利`**：每个单品在其最优价格档上的表现 $\pi_i^\*$，即贪心法的排序依据。
- **讲课提示**：**把连续定价离散化成价格档，是能用 MILP 的关键一步**。这个"连续问题离散化"的技巧适用性很广。

### `[45]` 路线 A：贪心 / 排序法
- **表 4-8**：SKU 数 N 从 27 扫到 33 的贪心结果。
- **讲课提示**：**只有基数约束时，可行集构成均匀拟阵（uniform matroid），而拟阵上最大化模函数，贪心恰好最优**。这是"什么时候可以放心用贪心"的经典判据，能讲清这点说明理解了算法而不只是调库。

### `[46]` 路线 B：混合整数线性规划
- **干什么**：PuLP + CBC 求解，含全部约束 C1–C6。
- **讲课提示**：MILP 的价值不是"跑得快"，而是**同时给出可行性与最优性证明**。

### `[47]` 路线 C：遗传算法
- **表 4-9 + 图 `30_求解对比`**：四条路线（贪心 / 纯基数 MILP / 含覆盖 MILP / GA）的目标值、耗时、最优性保证对比。
- **讲课提示**：GA 收敛到与 MILP 相同的目标值 → **交叉验证了解的可信度**。用两种独立算法互相印证，在论文里很有说服力。

### `[48]` 覆盖强度 $\alpha$ 权衡曲线
- **表 4-10 + 图 `31_覆盖权衡`**：$\alpha$ 从 0 扫到 1.3，看利润、SKU 数、总补货量如何变化，并找出**可行上限**。
- **讲课提示**：题目里"尽量满足市场对各品类蔬菜商品的需求"是一句模糊的话。**不要回避它，也不要过度承诺 100% 满足（数学上不可行）** —— 把它参数化成 $\alpha$ 并给出权衡曲线，这才是建模的态度。

### `[49]` ★ 把约束推到紧的一侧：贪心真的会翻车
- **干什么**：本课**最后一个方法论高潮**。在 $\alpha$ 很小时贪心和 MILP 打平，学生容易误以为"MILP 是多余的"。真正的差距必须在约束**紧**的地方才看得见。
- **表 4-10b**：$\alpha=0.9$ 时原始贪心解的 C6 校验 → **水生根茎类缺口 2.74 kg，不可行**。
- **表 4-10c + 图 `31b_贪心失效`**：三方案（原始贪心 / 贪心+人工修复 / MILP）对比。
- **最重要的读法**：**不可行的贪心解目标值反而更高（+1.08%）** —— 因为它是靠违反约束换来的。所以**比较算法必须先验可行性、再比目标值**；只贴目标值的对比表是没有意义的。反过来读，这多出来的利润就是约束 C6 的**影子价格**。
- **讲课提示**：本例中"贪心+修复"恰好也达到了 MILP 最优值，但要强调这是**运气不是保证** —— 修复启发式只承诺把解拉回可行域，没有任何机制证明最优。是 MILP 事后告诉我们"这次修对了"。

### `[50] [51]` ★ 问题三最终方案
- **表 4-11**：**问题三的答案** —— 2023-07-01 各单品的补货量与定价策略。
- **表 4-12**：品类汇总（验证 C5、C6 满足）。**表 4-13**：方案总计。**表 4-14**：★约束满足性校验（**论文必备**）。**表 4-15**：被淘汰单品及原因。
- **表 4-16 + 图 `32_问题三方案`**：SKU 数 N 的敏感性扫描（每个 N 都精确求解一次 MILP）。
- **交叉验证亮点**：问题二算出日均利润 **872 元**，问题三独立建模算出 7/1 单日 **874 元**，**两条互不相干的路线吻合到 0.2%**。这是模型可信度的有力证据，一定要在课上点出来。

## 第 5 部分　问题四与总结

### `[52]` 需求模型的方差分解
- **表 5-1/5-2 + 图 `33_方差分解`**：逐步加入变量看累计 $R^2$，量化各类信息的**边际解释力** $\Delta R^2$，以及"还有多少无法解释"。
- **讲课提示**：问题四不要空谈"建议采集更多数据"。**用残差说话**：模型解释不了的那部分有多大，就是信息缺口的量化证据。

### `[53] [54]` 数据需求清单与优先级
- **表 5-4**：数据需求按**维度 × 优先级**组织（而不是按数据类型罗列）。
- **图 `34_数据优先级`**：价值—难度四象限图，右上角"高价值 + 易采集"是立即该做的。
- **讲课提示**：建议要**可执行**。"采集天气数据"太空泛；"采集日最高温与降水量，用于解释叶菜类需求的短期波动"才有说服力。

### `[55]` 汇总导出
- **表 6-1**：全部结果表清单（45 个 CSV，位于 `outputs/`）。
- **讲课提示**：45 张表 + 38 张图可直接用于论文写作。

# 第六章　常见陷阱清单（考场提醒）

| # | 陷阱 | 正确做法 | 对应代码块 |
|---|---|---|---|
| 1 | 损耗率当成"卖不掉的比例" | 有效成本 $c=w/(1-\theta)$ | `[08]` |
| 2 | 残值取 $s=\delta p$（与决策变量挂钩） | $s=0$，否则积压成本为负 → 订货量发散 | `[06] [36]` |
| 3 | 用简单平均算当日价格 | 销量加权平均 | `[07]` |
| 4 | 删掉打折记录 | 保留销量，但价格另做处理 | `[05] [06]` |
| 5 | 直接 OLS 估价格弹性 | 价格内生 → 用差分或 IV | `[27]` |
| 6 | 只看 $R^2$ 选模型 | 滚动时序交叉验证 | `[28]` |
| 7 | 时序数据用普通 K-fold | rolling-origin（防数据泄漏） | `[28]` |
| 8 | 恒弹性模型直接解最优价 | $-1<\varepsilon<0$ 时无内点解，需可行域 | `[33] [34]` |
| 9 | 相关系数就下"品类互补"结论 | 做偏相关排除共同趋势 | `[17]` |
| 10 | 非平稳序列直接做 Granger | 先 ADF 检验 | `[24]` |
| 11 | 单品弹性直接用 OLS | 数据少 → 经验贝叶斯收缩 | `[42]` |
| 12 | 有品类约束还用贪心 | 破坏拟阵结构 → 必须 MILP | `[45] [49]` |
| 13 | 只比目标值不验可行性 | 不可行解目标值往往更高 | `[49]` |
| 14 | 服务水平顶在 99.9% 却不怀疑 | 边界解 + 扫描曲线全平 = 模型有问题 | `[36]` |
| 15 | 问题四空谈"多采集数据" | 用残差方差分解量化信息缺口 | `[52]` |

# 第七章　两小时时间分配建议

| 时段 | 内容 | 重点 |
|---|---|---|
| 0–10 min | 题目背景、四问关系、建模总蓝图 | `[01]` 那张图 |
| 10–25 min | 四份附件逐字段讲解、数据工程 | **`[06]` 打折盈亏检验必讲** |
| 25–50 min | 问题一七种方法 | 重点 `[17]` 偏相关、`[24]` ADF+Granger |
| 50–85 min | 问题二：需求估计 → 成本预测 → 报童优化 | **`[27]` 规格比较、`[36]` 永动机陷阱必讲** |
| 85–110 min | 问题三：收缩 + MILP + 三算法对比 | **`[49]` 贪心翻车必讲** |
| 110–120 min | 问题四 + 陷阱清单 + 论文写作建议 | 陷阱清单发给学生带走 |

**如果时间被压缩到 1 小时**，保留这四个片段即可覆盖全部方法论要点：
`[06]` 用数据推翻假设 → `[27]` 模型设定诊断 → `[36]` 识别退化解 → `[49]` 可行性优先于目标值。